#### ***Query Optimization***



##### **Query Optimmization means improve the user's original query into a better query,then rerank can find better context.**

#### ***Why Query Optimization***

##### **The problem is that users don't always write retrieval-friendly queries.**

In [36]:
### load the Environment Variables

from dotenv import load_dotenv
load_dotenv(override=True)

True

In [37]:
### Check/Validate the path to load the files from the Document
import os

path = "../kubernetes"

if os.path.exists(path):
    print("Path is valid")
else:
    print("Path is not valid")

Path is valid


In [38]:
#load the documents using DirectoryLoader 

from langchain_community.document_loaders import DirectoryLoader,PyMuPDFLoader

loader = DirectoryLoader(
    path,
    glob = "*.pdf",
    loader_cls=PyMuPDFLoader
)

documents = loader.load()

print("Number Of Documents:",len(documents))

Number Of Documents: 3983


In [39]:
### create a chunks using splitter

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700,
    chunk_overlap = 100
)

chunks = splitter.split_documents(documents)

print("Number Of Chunks:",len(chunks))

Number Of Chunks: 12694


In [40]:
###BM25 Retriever
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(chunks)

bm25_retriever.k=10

In [41]:
####Initalize the embedding model

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model = "BAAI/bge-large-en-v1.5",
    model_kwargs = {"device":"cpu"},
    encode_kwargs = {"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [42]:
### load the vcctors from Chroma DB
from langchain_chroma import Chroma

vectorstore = Chroma(
    persist_directory="../vectorstore/kubernetes_rag",
    collection_name="kubernetes_rag",
    embedding_function=embedding_model
)


In [43]:
vectorstore._collection.count()

12694

In [44]:
#### retriever for similarity search.
vector_retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":10}
)

In [45]:
### Hybrid Search
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever,bm25_retriever],
    weights = [0.7,0.3]
)

In [46]:
### Lets test with hybrid retriever and how many candidates are retrieved
docs = hybrid_retriever.invoke("What is Kubernetes Deployment?")
print(len(docs))

20


In [47]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [48]:
### Get top 5 final document after reranking

def retrieve_and_rerank(query, k=5):

    retrieved_docs = hybrid_retriever.invoke(query)


    pairs = [[query,doc.page_content] for doc in retrieved_docs]

    scores = reranker.predict(pairs)

    ##zip the scores and retrieved order the documents based on score from highest to lowest

    ranked_docs = sorted(zip(retrieved_docs,scores),key = lambda x:x[1],reverse=True)

    ## Get top k documents using ranked_docs and k value
    top_docs = [doc for (doc,score) in ranked_docs[:k]]

    #for i,(doc,score) in enumerate(ranked_docs[:k],1):
        #print(f"""
       #         {i}  | Page:{doc.metadata.get("page")} | Score:{score}
        #""")

    return top_docs

In [49]:
### Format without unnecessary context

def build_context(documents):

    context = ""

    for i,doc in enumerate(documents,start=1):
        source = doc.metadata.get("source")
        source = source.replace("\\","/")
        source = source.split("/")[-1]
        #print(source)
        context+=f"""
        
    Source {source}
    Page: {doc.metadata.get("page")}
    Content: {doc.page_content}
        """
    return context



In [50]:
### Design a prompt
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
You are a question-answering assistant.

Answer the question using ONLY the provided context.

Rules:
1. Do not use outside knowledge.
2. Do not invent information.
3. If the context does not contain the answer, say:
   "I don't have information based on the provided documents."
4. Keep the answer concise and directly relevant.
5. List the sources and page numbers used at the very end of your response.
6. Format the sources strictly like this:
[Source: filename, Page: number]

Context:
{context}

Question:
{question}

Answer:
""")

In [51]:
### Create LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b"
)

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000029F13C42CC0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000029F13C406E0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [52]:
query = "What is Kubernetes Deployment?"

top_5_chunks = retrieve_and_rerank(query)

for i,chunk in enumerate(top_5_chunks,start=1):
    print(f"------ Document {i} --------")
    print("Page:",chunk.metadata.get("page"))

------ Document 1 --------
Page: 9
------ Document 2 --------
Page: 5
------ Document 3 --------
Page: 638
------ Document 4 --------
Page: 6
------ Document 5 --------
Page: 12


In [53]:
## Change the project name for Langsmith observability.
from langchain_core.tracers import LangChainTracer

custom_tracer = LangChainTracer(project_name="Generation Evaluation For Kubernetes RAG.")

In [54]:
config = {"run_name":"Generation Evaluation","callbacks":[custom_tracer]}

In [ ]:
def final_rag_response(query):

    top_5_docs = retrieve_and_rerank(query)

    context = build_context(top_5_docs)

    #print("Context:",context)

    messages = prompt.invoke({"question":query,"context":context})

    response = llm.invoke(messages,config=config).content

    return response

#### ***Build the Query Optimizer***

In [56]:
from langchain_core.prompts import ChatPromptTemplate

query_optimizer_prompt = ChatPromptTemplate.from_template("""
You are a query optimization component for a Kubernetes documentation RAG system.

Rewrite the user's query into a clear, specific search query
that will help retrieve relevant documents.

Rules:
- Preserve the original intent.
- Add important context when it is implied by the query.
- Use precise technical terminology.
- Do not answer the question.
- Do not add information that changes the user's intent.
- Return only the optimized query.

User query:
{query}
""")

In [57]:
query_optimizer = query_optimizer_prompt | llm

In [58]:
query = "What happens if one goes down?"

In [59]:
optimized_query = query_optimizer.invoke({"query":query})
print("Original Query:",query)
print("Optimized Query:",optimized_query.content)

Original Query: What happens if one goes down?
Optimized Query: Kubernetes behavior and workload handling when a single node fails or goes down.


In [60]:
queries = [
    "What is a Kubernetes Deployment?",
    "What is a Kubernetes Pod?",
    "What is a Kubernetes Service?",
    "What is a ReplicaSet?",
    "How does a Deployment manage ReplicaSets during an update?",
    "How does a Service identify which Pods should receive traffic?"
]
for query in queries:
    optimized_query = query_optimizer.invoke({"query":query})
    print("Original Query:",query)
    print("Optimized Query:",optimized_query.content)

Original Query: What is a Kubernetes Deployment?
Optimized Query: Kubernetes Deployment resource definition and purpose overview.
Original Query: What is a Kubernetes Pod?
Optimized Query: Kubernetes Pod definition and purpose documentation.
Original Query: What is a Kubernetes Service?
Optimized Query: Kubernetes Service object definition and purpose.
Original Query: What is a ReplicaSet?
Optimized Query: Kubernetes ReplicaSet definition, purpose, and functionality documentation.
Original Query: How does a Deployment manage ReplicaSets during an update?
Optimized Query: Kubernetes Deployment rolling update behavior and management of ReplicaSets during updates.
Original Query: How does a Service identify which Pods should receive traffic?
Optimized Query: Kubernetes Service selector mechanism: how a Service matches Pods (label selectors, endpoints) to route traffic.


#### ***Original vs Optimized Retrieval***

### RAG with larger, more realistic questions.

In [62]:
queries = [
    "How do Deployments, ReplicaSets, and Pods work together to maintain the desired number of application instances, and what happens when a Pod fails?",

    "How does a Kubernetes Deployment perform a rolling update, including how it creates and scales ReplicaSets and how it maintains application availability during the update?",

    "How does a Kubernetes Service discover Pods created by a Deployment, and how are label selectors, EndpointSlices, ClusterIP, and load balancing involved in routing traffic?",

    "What happens in Kubernetes when a Pod managed by a Deployment is deleted or fails, and which controllers are responsible for detecting the failure and creating a replacement?",

    "How do multiple containers within the same Pod communicate and share resources, and what networking, storage, and IPC mechanisms are available to them?",

    "Explain the complete relationship between a Deployment, ReplicaSet, Pod, and Service when deploying and exposing a scalable application in Kubernetes.",

    "How does Kubernetes continuously reconcile the desired state of an application with the actual state, using Deployments, ReplicaSets, Pods, and Services as examples?"
]

for query in queries:
    print("="*60)
    print("Query:",query)
    print("="*60)
    response = final_rag_response(query)
    print("Response:",response)

Query: How do Deployments, ReplicaSets, and Pods work together to maintain the desired number of application instances, and what happens when a Pod fails?
Response: Deployments let you declare the desired number of Pods. When you create a Deployment it automatically creates a ReplicaSet that owns those Pods. The ReplicaSet continuously watches the Pods that match its selector and keeps the count of running Pods equal to the replica number you specified, so the application’s required number of instances is always maintained.  

If a Pod is deleted, crashes, or is lost because a node fails, the ReplicaSet notices the shortfall and creates a replacement Pod, keeping the total number of healthy Pods at the desired level. Using a Deployment (instead of “naked” Pods) ensures this automatic replacement and continuous availability.  

**Sources**  
[Source: Concepts.pdf, Page: 130]  
[Source: Concepts.pdf, Page: 156]  
[Source: Concepts.pdf, Page: 164]  
[Source: Concepts.pdf, Page: 221]  
[So

In [63]:
test_queries = [
    "What happens when a Deployment specifies three replicas?",
    "What happens if one of the Pods managed by a Deployment fails?",
    "How does Kubernetes ensure that the actual state matches the desired state?",
    "How does a Service identify which Pods should receive traffic?",
    "How do containers within the same Pod communicate with each other?",
    "What is the difference between a Pod and a Deployment?",
    "What is the difference between a Deployment and a ReplicaSet?",
    "How are Deployments, ReplicaSets, and Pods related?",
    "How does a Deployment use ReplicaSets to manage Pods?",
    "How does a Service provide access to Pods managed by a Deployment?",
    "How can I run multiple copies of the same application?",
    "How can other applications find and communicate with my Pods?",
    "If I want three copies of my application running, which Kubernetes object should I use?",
    "If a Pod is deleted manually, how does Kubernetes respond when it is managed by a ReplicaSet?",
    "Which Kubernetes component continuously works to reconcile desired state and actual state?"
]
for query in test_queries:
    print("="*60)
    print("Query:",query)
    print("="*60)
    response = final_rag_response(query)
    print("Response:",response)

Query: What happens when a Deployment specifies three replicas?
Response: When a Deployment’s spec sets `replicas: 3`, Kubernetes tries to run three instances of the application – it creates three Pods (or attempts to) and continuously works to keep the actual state matching that desired count.  

[Source: Concepts.pdf, Page: 5]  
[Source: Tasks.pdf, Page: 103]
Query: What happens if one of the Pods managed by a Deployment fails?
Response: If a Pod that is part of a Deployment fails, the Deployment’s controller detects the failure and automatically creates a replacement Pod, which the scheduler then places on a healthy node.  

[Source: Concepts.pdf, Page: 87]
Query: How does Kubernetes ensure that the actual state matches the desired state?
Response: Kubernetes keeps the actual state in sync with the desired state by using a declarative model and continuous reconciliation loops. You declare the desired state in an object’s spec (e.g., the number of replicas in a Deployment). The contr

In [64]:
queries = [
    "What is a Kubernetes Deployment?",
    "How are Deployments, ReplicaSets, and Pods related?",
    "What happens if a Pod managed by a ReplicaSet is deleted?",
    "How does a Kubernetes Service identify which Pods should receive traffic?",
    "How does a Deployment use ReplicaSets to maintain the desired number of Pods during a rolling update?"
]
for query in queries:
    print("="*60)
    print("Query:",query)
    print("="*60)
    response = final_rag_response(query)
    print("Response:",response)

Query: What is a Kubernetes Deployment?
Response: A Kubernetes Deployment is a Kubernetes object that defines the desired state for an application—specifying how many pod replicas should run, how they should be created and updated, and how they should be monitored. The Deployment controller continuously ensures that the actual pods match this specification, automatically replacing pods that fail or are deleted and handling scaling and updates. [Source: Tutorials.pdf, Page: 9] [Source: Concepts.pdf, Page: 5] [Source: Tutorials.pdf, Page: 2]
Query: How are Deployments, ReplicaSets, and Pods related?
Response: Deployments are a higher‑level controller that own and manage ReplicaSets; the Deployment creates a ReplicaSet, and that ReplicaSet in turn creates and maintains the Pods. Deployments provide declarative, server‑side updates for Pods by controlling the ReplicaSets they own, while ReplicaSets handle the actual Pod creation, deletion and replacement. [Source: Concepts.pdf, Page: 130] 

### **Based on few test Query Optimization is not required for our Rag System**